# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Method: Logistic Regression first (readable baseline model), then Random Forest (stronger,
still explainable via feature importance). This fits a yes/no-with-observed-label question
(is_declining_label = trend_direction == 'down'). I'm not using clustering or gradient
boosting -- my question is "will this page decline," which is squarely a classification/
ranking task, not a grouping task, and the starter pipeline's own results show Random Forest
already beats simpler methods meaningfully, so added boosting complexity isn't earning its
keep yet.

I also fixed my Week-4 baseline: baseline_score_v2 scores declining pages by impressions
ALONE (no longer multiplying the label directly into the score), because the original
baseline_score = declining * has_demand * impressions guaranteed Precision@50 = 1.000 by
construction -- the label was literally the score, which is a leakage-flavored problem, not
a real result. This new version keeps the baseline honest.

In [1]:
import os
if not os.path.exists('/content/flyrank-ml'):
    !git clone https://github.com/SuryaK5125/flyrank-ml.git
%cd /content/flyrank-ml
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# FIX the circular baseline from w04: score by impressions among declining pages only,
# without multiplying the label into the score itself
declining = (df['trend_direction'] == 'down').astype(int)
df['baseline_score_v3'] = df['impressions_90d']
# still correlated with the label (declining pages score >0), but NOT guaranteed precision=1.0
# since among declining pages, ranking by impressions alone doesn't guarantee top-K are labeled correctly
# vs a random ordering of declining pages -- this is a fairer, though still simple, baseline

/content/flyrank-ml


## 2. Split design

Split design: grouped by client_id using GroupShuffleSplit, 70/30. A random row-level split
would risk pages from the same client appearing in both train and test, letting the model
partially memorize client-specific patterns instead of learning generalizable signal -- this
is the exact "client/group holdout" warning from the lane guide. The overlap check above
confirms zero shared clients between train and test.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['search_volume','competition','cpc','word_count','char_count',
                 'impressions_90d','clicks_90d','sessions_90d','content_age_days',
                 'days_since_last_update','ctr','avg_position','engagement_rate',
                 'scroll_rate']
target_col = 'trend_direction'

model_df = df.dropna(subset=feature_cols + [target_col, 'client_id']).copy()
model_df['label'] = (model_df[target_col] == 'down').astype(int)

# GROUP split by client_id -- pages from the same client must not appear in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]

print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test: {len(test_df)} rows, {test_df['client_id'].nunique()} clients")
print(f"Overlap check (should be empty set): {set(train_df['client_id']) & set(test_df['client_id'])}")

Train: 14114 rows, 20 clients
Test: 5783 rows, 9 clients
Overlap check (should be empty set): set()


## 3. Train + compare vs my baseline

Base rate (random guessing): 0.623
Fixed baseline (rank by impressions_90d alone): 0.520 -- actually WORSE than random. This
makes sense once you think about it: high-impression pages are just as likely to be stable
or growing as declining, so sorting by impressions alone doesn't separate decliners from
non-decliners at all. My Week-4 baseline's 1.000 precision was an illusion caused by the
label being multiplied directly into the score -- this is the honest number for a simple,
non-circular rule.

Logistic Regression: 0.720
Random Forest: 0.720
Both models clearly beat the base rate (+15.6%) and the honest baseline (+38.5%) at
Precision@50. This is a real, meaningful lift -- ranking by a learned combination of signals
genuinely separates declining pages from stable/growing ones better than either random
selection or a single-signal rule.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_train, y_train = train_df[feature_cols], train_df['label']
X_test, y_test = test_df[feature_cols], test_df['label']

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42).fit(X_train, y_train)

logreg_scores = logreg.predict_proba(X_test_s)[:,1]
rf_scores = rf.predict_proba(X_test)[:,1]

# baseline scored on the SAME test set
baseline_test_scores = test_df['baseline_score_v3'].values
base_rate = y_test.mean()

results = pd.DataFrame({
    'method': ['base_rate (random)', 'baseline_v3 (rule)', 'logistic_regression', 'random_forest'],
    'precision_at_50': [
        base_rate,
        precision_at_k(baseline_test_scores, y_test.values, 50),
        precision_at_k(logreg_scores, y_test.values, 50),
        precision_at_k(rf_scores, y_test.values, 50),
    ]
})
print(results)

                method  precision_at_50
0   base_rate (random)          0.62286
1   baseline_v3 (rule)          0.52000
2  logistic_regression          0.72000
3        random_forest          0.72000


## 4. Errors and interpretation

Top features (Random Forest): impressions_90d (0.270), avg_position (0.137),
content_age_days (0.132), clicks_90d (0.076), ctr (0.059). This makes intuitive sense --
impressions and position are the most direct measures of current search visibility, and
age plausibly correlates with decline risk since older content is more likely to have been
overtaken. None of these look suspiciously dominant (no single feature above ~0.30), which
is a good sign against leakage -- a leaked feature usually dominates far more sharply,
closer to what I saw in the w03 leak-trap exercise.

Where the model is wrong: the 3 false-negative examples show the model missing real declines
even with meaningfully different profiles -- one has moderate impressions (687) and modest
position (14.2), one has very high impressions (167,858) and a strong position (5.1) yet
still declined, and one has almost no impressions (2) and a weak position (50.0). The high-
impression, strong-position miss is the most interesting case: a page that looks healthy by
every visible signal but is still losing ground -- exactly the kind of decline a simple rule
would never catch, and even the model under-scores it (0.285 probability). That's a
legitimate limitation worth flagging rather than hiding.

Model vs baseline: Random Forest and Logistic Regression tie at Precision@50 = 0.720, both
meaningfully ahead of the fixed baseline (0.520) and base rate (0.623). ROC-AUC is modest
(0.586 / 0.612) -- lower than the starter pipeline's reported numbers, likely because I used
a grouped-by-client split (stricter, more honest generalization test) rather than a random
row split.

In [4]:
from sklearn.metrics import roc_auc_score
print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_test, logreg_scores):.3f}")
print(f"Random Forest ROC-AUC: {roc_auc_score(y_test, rf_scores):.3f}")

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop features (Random Forest):")
print(importances.head(5))

# 3 concrete wrong cases
test_df_copy = test_df.copy()
test_df_copy['rf_pred_prob'] = rf_scores
wrong = test_df_copy[(test_df_copy['label']==1) & (test_df_copy['rf_pred_prob'] < 0.3)]
print(f"\n3 false-negative examples (actually declining, model scored low):")
print(wrong[['content_id','impressions_90d','avg_position','rf_pred_prob']].head(3))

Logistic Regression ROC-AUC: 0.586
Random Forest ROC-AUC: 0.612

Top features (Random Forest):
impressions_90d     0.269975
avg_position        0.137350
content_age_days    0.132292
clicks_90d          0.076223
ctr                 0.058888
dtype: float64

3 false-negative examples (actually declining, model scored low):
                content_id  impressions_90d  avg_position  rf_pred_prob
491   content_b65c621c6c0b              687          14.2      0.295048
1448  content_62ed76850efc           167858           5.1      0.285053
1864  content_16f38acf0f26                2          50.0      0.160217


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.